# Take-Home Case: The Mortgage Book of Aare-Säntis Regionalbank AG

**EAIF: AI for Finance — team take-home between the classes on linear/logistic regression and on advanced supervised learning**

You are the newly formed data team of Aare-Säntis Regionalbank AG, a fictional regional bank in the Swiss Mittelland. The bank has 10,000 residential mortgages on its books, originated between 2019 and 2022, spread over seven cantons. Until now, the bank's credit decisions have rested on a vendor valuation model, a handful of ratio rules and the judgement of the branch advisers. This morning the Chief Risk Officer sent the team its first assignment.

> **Memo from the CRO**
>
> To the data team. Welcome aboard. Three things, in order of urgency.
>
> 1. Our collateral valuations come from a vendor model we cannot inspect. Build one we can.
> 2. About seven in a hundred of our recent mortgages ran into payment trouble within three years. Tell me which applications carry that risk, and where we should set the approval bar, in francs.
> 3. The risk committee has read that banks use "machine learning" for this. Show me whether the more flexible methods actually do better on our book, and whether we could defend deploying one.
>
> I do not need a slide deck. I need numbers I can trust and one paragraph per question that I can read out to the committee.
>
> Head of Risk

This notebook is your working file for the assignment. Part A answers the first two requests with the methods you already know, linear and logistic regression. Part B answers the third with the methods introduced in the next class. Every section ends with what you, the analyst, tell the CRO.

## The data

Two files are loaded from the course repository. `mortgages.csv` is the bank's book: 10,000 mortgages originated 2019 to 2022, each with its outcome after 36 months. `applications_2026.csv` holds this year's 500 applications, with the same columns but no outcome, because the bank has not decided on them yet. The full column dictionary is in [`data/mortgage2026/README.md`](https://github.com/umatter/EDFB/blob/main/data/mortgage2026/README.md); the groups of columns are these.

| Group | Columns |
|---|---|
| Property | `canton`, `property_type`, `living_area_m2`, `rooms`, `year_built`, `distance_center_km`, `energy_label`, `purchase_price` |
| Borrower | `household_income`, `age`, `employment`, `years_client` |
| Loan | `loan_amount`, `rate_type`, `fixed_years`, `interest_rate`, `amortisation`, `origination_year` |
| Derived ratios | `ltv`, `affordability`, `actual_burden` |
| Outcome | `trouble_36m` (1 if the mortgage was 90 days or more in arrears, or was restructured, within 36 months) |

Three ratios do most of the work in Swiss mortgage lending, and all three are in the file.

The loan-to-value ratio compares the loan with the price of the property: `ltv = loan_amount / purchase_price`. Swiss banks normally finance at most 80 % of the price, and the part of the loan above two-thirds of the price must be amortised within 15 years.

The affordability ratio is the Swiss lending rule for whether the household can carry the loan through a rise in interest rates. It does not use the interest rate actually agreed but an imputed rate of 5 %, adds 1 % of the purchase price per year for maintenance, adds the amortisation of the part above two-thirds LTV spread over 15 years, and divides the sum by gross household income:

`affordability = (0.05 × loan_amount + 0.01 × purchase_price + amortisation per year) / household_income`

The rule says the ratio must not exceed one third. A household with CHF 150,000 gross income and a CHF 800,000 loan on a CHF 1,000,000 property carries 40,000 of imputed interest, 10,000 of maintenance and about 8,900 of amortisation per year, which is 0.39 of its income, above the bar.

The actual burden is what the household pays in interest today, at the agreed rate: `actual_burden = interest_rate × loan_amount / household_income`. With rates in the range of 1 to 2 %, it is a fraction of the affordability ratio, which is exactly why the imputed rate exists.

> **Simulated data.** The two files were generated by the course for this case. There is no real bank, property or household behind any row. The rate of payment trouble in the book is several times higher than in a real Swiss mortgage book so that the models have enough cases to learn from; the ratios, prices and incomes are in a realistic range but were not taken from any real data source.

One more thing to know before you start: the book contains one column that the bank does *not* know at the moment it decides on an application. Part A2 deals with it.

## How to work through this notebook

Plan three hours for a team of three to four. Make a copy of the notebook in your Drive (File, Save a copy in Drive) and work in the copy. Run the given cells one after the other and read them, including the comments; they are the worked part of the case and they are where the methods are explained. The exercises are marked `### Exercise n` and each one is followed by an empty code cell. Fill that cell and leave the given cells as they are, because later parts of the notebook use the objects they create.

Rotate who types for each part, so that nobody sits through the whole case as a spectator. In the next class, any member of the team can be asked to explain any cell, worked or exercise, so make sure everyone can. The notebook is discussed in that class; it is not graded.

Every choice of columns in this case follows one rule, which you will meet again in every part:

**At the moment the bank decides on an application, which columns are already known?**

Everything a model uses must pass that test. A model that is fed a column the bank only learns afterwards looks excellent in the notebook and is useless at the counter.

## Roadmap

| Part | Question | Method | Exercises |
|---|---|---|---|
| A1 | Is our collateral valued right? | Linear regression | 1, 2 |
| A2 | Which applications will run into payment trouble, and where is the approval bar in francs? | Logistic regression | 3, 4 |
| B | Do the flexible methods do better on our book? | LASSO, decision tree, random forest, gradient boosting and XGBoost, SVM, comparison | |
| C | Your turn on the new methods | The methods of Part B, and reflection questions for the class | 5 to 8 |

## Setup

In [ ]:
# Setup: Colab's preinstalled stack only, nothing to install
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance, DecisionBoundaryDisplay
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, roc_auc_score,
                             roc_curve, confusion_matrix, precision_score, recall_score)
import xgboost as xgb

np.random.seed(0)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")

# The two error costs the CRO gave us, in CHF, used throughout the notebook
COST_FN = 60_000   # a loan that runs into trouble was approved: expected loss
COST_FP = 8_000    # a loan that would have been fine was rejected: lost margin

print("Setup complete. scikit-learn", __import__("sklearn").__version__, "| xgboost", xgb.__version__)

In [ ]:
# Load the two data files from the course repository
DATA = "https://raw.githubusercontent.com/umatter/EDFB/main/data/mortgage2026/"
book = pd.read_csv(DATA + "mortgages.csv")
apps = pd.read_csv(DATA + "applications_2026.csv")
print("book:", book.shape, "| applications:", apps.shape)
book.head()

In [ ]:
# Types, missing values, and the outcome's base rate
print(book.dtypes, "\n")
print("missing values:", int(book.isna().sum().sum()))
print(f"trouble rate in the book: {book.trouble_36m.mean():.3%}  ({book.trouble_36m.sum()} of {len(book)})")
book.describe().T

# Part A: Recap

## A1. Is our collateral valued right? Linear regression

The bank lends against the property. If the borrower stops paying, the bank sells the property and recovers what it can, so the question behind every mortgage is what the property is worth, as opposed to what the buyer paid for it. Today that question is answered by a vendor model that returns a number and no explanation. The CRO's first request is a valuation the bank can inspect.

The standard tool for this is a hedonic model: the price of a property is written as the sum of the prices of its characteristics. A linear regression is exactly such a model, and it is inspectable by construction. Its coefficients are a price per square metre, a premium or discount per canton, a discount per kilometre from the regional centre, a premium for a house over an apartment. A valuer can read those numbers, argue with them, and compare them with what the market pays.

We follow the steps of the first supervised-learning class.

1. Look at the data, in a plot, before fitting anything.
2. Pick the target Y (`purchase_price`) and the features X (the property columns).
3. Split the book into a training set and a test set.
4. Fit the model on the training set.
5. Test it on the held-out data, and put its RMSE next to the RMSE of a baseline that knows nothing.

The features are all property characteristics, which the bank knows when the application arrives, so the decision-time rule is satisfied.

In [ ]:
# Step 1: look at the relationship we want to model. Price against living area, one point per mortgage.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
ax.set_xlabel("living area (m²)")
ax.set_ylabel("purchase price (CHF million)")
ax.set_title("The bank's book: purchase price against living area")
plt.show()

In [ ]:
# Step 2: the simplest model. One X, one slope: CHF per square metre, averaged over everything else.
uni = LinearRegression().fit(book[["living_area_m2"]], book.purchase_price)
print(f"price = {uni.intercept_:,.0f} + {uni.coef_[0]:,.0f} x living area")
print(f"R² on the full book: {uni.score(book[['living_area_m2']], book.purchase_price):.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(book.living_area_m2, book.purchase_price / 1e6, s=6, alpha=0.3)
grid = np.linspace(45, 320, 50).reshape(-1, 1)
ax.plot(grid, uni.predict(pd.DataFrame(grid, columns=["living_area_m2"])) / 1e6, color="C3", lw=2)
ax.set_xlabel("living area (m²)"); ax.set_ylabel("purchase price (CHF million)")
ax.set_title("Simple linear regression: one slope for all cantons")
plt.show()

The slope is the whole model. In this run it says that one additional square metre of living area adds CHF 7,589 to the price, averaged over every canton, every building age and every location in the book, and living area alone explains 48 % of the variation in prices (an R² of 0.484). The line runs through the middle of the cloud, but the cloud is wide: at 150 m² the book holds properties that sold for well under one million and others that sold for close to two.

One slope is not enough because a square metre does not cost the same everywhere. A square metre in the canton of Zurich and one in the canton of Solothurn are priced in different markets. A house built in 1960 and one built in 2018 differ in what a buyer pays, as do a flat next to the station and one twenty kilometres out. The simple model averages over all of that, and the spread around the line is the price of averaging.

The multivariate model puts these characteristics in as further columns of X. The numeric ones (`living_area_m2`, `year_built`, `distance_center_km`) enter as they are. The categorical ones (`canton`, `property_type`, `energy_label`) become 0/1 dummy columns, one per level, with one reference level dropped per column, exactly as in the logistic-regression class: the coefficient of each dummy is then the premium relative to the dropped level. `pd.get_dummies(..., drop_first=True)` drops the alphabetically first level, so the references are canton AG, property type `apartment` and energy label A.

In [ ]:
# Step 3: the multivariate model. Categorical columns become 0/1 dummies (one reference level dropped each).
price_features = ["canton", "property_type", "living_area_m2", "year_built", "distance_center_km", "energy_label"]
Xp = pd.get_dummies(book[price_features], drop_first=True).astype(float)
yp = book.purchase_price

Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.3, random_state=42, stratify=book.canton)
lin = LinearRegression().fit(Xp_train, yp_train)

coef = pd.Series(lin.coef_, index=Xp.columns).sort_values()
print("intercept:", f"{lin.intercept_:,.0f}")
coef.round(0).to_frame("CHF per unit")

Every row of the table is a price the bank can read. In this run, one square metre of living area adds CHF 6,705, now holding canton, building age, location and energy label fixed, which is about CHF 900 less than the slope of the simple model. The canton dummies are premiums relative to Aargau, the dropped reference: a property in the canton of Zurich sells for about CHF 403,000 more than the same property in Aargau, one in Solothurn for about CHF 187,000 less. Each kilometre further from the regional centre takes about CHF 17,900 off the price. A house sells for about CHF 96,000 more than an apartment with the same area, age, canton and location, and each energy label below A carries its own discount.

These numbers are the inspectable model the CRO asked for. A valuer who disagrees with the Zurich premium or the discount per kilometre can say so, in francs, and the bank can check the number against recent transactions.

In [ ]:
# Step 4: test the model on data it has not seen, next to the baseline "predict the training mean"
def rmse_of(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

pred_test = lin.predict(Xp_test)
baseline = np.full(len(yp_test), yp_train.mean())
print(f"RMSE  linear model : CHF {rmse_of(yp_test, pred_test):>10,.0f}   (R² {r2_score(yp_test, pred_test):.3f})")
print(f"RMSE  predict mean : CHF {rmse_of(yp_test, baseline):>10,.0f}   (R² {r2_score(yp_test, baseline):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(yp_test / 1e6, pred_test / 1e6, s=6, alpha=0.3)
lim = [0, yp_test.max() / 1e6]
axes[0].plot(lim, lim, color="C3", lw=1)
axes[0].set_xlabel("actual price (CHF million)"); axes[0].set_ylabel("predicted price (CHF million)")
axes[0].set_title("Test set: predicted against actual")
axes[1].scatter(pred_test / 1e6, (yp_test - pred_test) / 1e6, s=6, alpha=0.3)
axes[1].axhline(0, color="C3", lw=1)
axes[1].set_xlabel("predicted price (CHF million)"); axes[1].set_ylabel("residual (CHF million)")
axes[1].set_title("Residuals fan out as the price grows")
plt.show()

The test set holds 3,000 mortgages the model has never seen. In this run the linear model's RMSE on the test set is CHF 154,786, against CHF 426,031 for the baseline that predicts the training mean for every property, and it explains 87 % (an R² of 0.868) of the variation in test prices where the baseline explains none. For a property that the model values at CHF 900,000, an error of one RMSE means that the true market value is plausibly anywhere between about CHF 745,000 and CHF 1,055,000, which is the uncertainty the bank has to keep in mind when it sets the loan-to-value ratio.

The right-hand plot shows something the RMSE hides. The residuals are not a band of constant width; they fan out. Cheap properties are missed by a small amount, expensive ones by a large amount, and the error grows in proportion to the price. That is the signature of a multiplicative relationship: a Zurich premium is more naturally a percentage of the price than a fixed sum in francs, and so is the discount for an old building. A model in levels cannot express that, which is why it is imprecise exactly where the bank's exposures are largest. Exercise 1 takes the hint.

What the analyst tells the CRO: a linear regression on six property characteristics values the collateral to within about CHF 155,000 on unseen properties, every coefficient is a price in francs that a valuer can inspect and challenge, and the model's main weakness, its imprecision for expensive properties, can be addressed by modelling the price in logarithms, which is the next step.

### Exercise 1: A model in logarithms

Fit the same multivariate model on `np.log(yp_train)` instead of `yp_train`. Predict on the test set, transform the predictions back with `np.exp`, and report the RMSE in CHF next to the RMSE of the model in levels. Then plot the residuals of the log model against its predictions as in the cell above.

*Deliverable:* the two RMSE values, and one sentence on which model the bank should use and why (look at the residual plot, not only at the RMSE).

In [ ]:
# Exercise 1: your code here
# lin_log = ...

### Exercise 2: Did the buyer overpay? A feature for Part B

Use your log model to predict a value for **every** property in the book (`Xp`, not only the test set) and compute `overpayment = purchase_price / predicted value`. A value above 1 means the buyer paid more than comparable properties cost. Store it as `book["overpayment"]`. Then compare the trouble rate of the mortgages in the top 10 % of `overpayment` with the rest.

*Deliverable:* the two trouble rates and one sentence on whether the valuation model tells the bank something about repayment risk. (Part B recomputes this column itself, so the notebook keeps running if you skip this.)

In [ ]:
# Exercise 2: your code here
# book["overpayment"] = ...